# Descargar Video del Partido

Baja el video de un partido de **lpfplay.com** (immergo.tv, HLS sin DRM) y lo
guarda en `data/temporadas/{año}/partidos/{match_key}/video/`, listo para
`analisis_video.ipynb`.

**No hace falta Chrome ni tu sesión.** La página del player pide member para
reproducir, pero el endpoint de metadata (`insight-api-shared.univtec.com`)
expone igual el `guid` del video, y el archivo HLS en `lpf.vod.immergo.tv` no
tiene ninguna protección (`hasDrm:false`, sin geoblock, sin cookies) — se
descarga directo con `requests` + `yt-dlp`.

> Solo descargar contenido para el que tengas derecho de uso. Es para análisis
> propio del equipo.

In [1]:
import os, sys

%load_ext autoreload
%autoreload 2

if os.path.basename(os.getcwd()) in ('primera_nacional', 'liga_profesional'):
    os.chdir('..')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import config

# ============================================================
# CONFIGURAR ACA
PAGE_URL  = 'https://www.lpfplay.com/player/episode/6a9dd6070fcb4d947e0c7751'
SEASON    = '2026'
MATCH_KEY = '2026-09-06_vs_estudiantes_caseros_h'  # YYYY-MM-DD_vs_<rival>_<h|a>
QUALITY   = '1080p'   # '360p' | '720p' (recomendado CV) | '1080p'

# Escape hatch: si lpfplay cambia el endpoint y esto deja de andar, pega acá
# directamente el .m3u8 (DevTools F12 > Network > filtro 'm3u8').
MANIFEST_MANUAL = ''
# ============================================================

VIDEO_DIR = os.path.join(config.season_data_dir(SEASON), MATCH_KEY, 'video')
print(f'Partido : {MATCH_KEY}')
print(f'Calidad : {QUALITY}')
print(f'Destino : {VIDEO_DIR}')

Partido : 2026-09-06_vs_estudiantes_caseros_h
Calidad : 1080p
Destino : c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\data\temporadas\2026\partidos\2026-09-06_vs_estudiantes_caseros_h\video


## El descargador (incrustado)

`get_master_manifest()` saca el `episode_id` del link del player, le pega al
endpoint de metadata de univtec (con headers de dispositivo inventados, sin
cookies ni login) y arma la URL del `master.m3u8` a partir del `guid` que
devuelve. `download_match_video()` baja ese HLS con yt-dlp.

**Por qué la versión con Chrome no andaba:** la página del player pide una
membresía (`commerceType:"member"`) para reproducir, así que el reproductor se
queda esperando el anuncio/login para siempre y nunca pide el `.m3u8`. Pero el
archivo de video en sí (`lpf.vod.immergo.tv`) no tiene esa restricción —
alcanza con saber su `guid`, y el endpoint de metadata lo entrega igual, gratis.

In [2]:
import re, uuid


def _episode_id(page_url_or_id):
    m = re.search(r'episode/([a-f0-9]{24})', page_url_or_id)
    return m.group(1) if m else page_url_or_id


def get_master_manifest(page_url_or_id, verbose=True):
    """Resuelve el .m3u8 de un episodio de lpfplay SIN Chrome ni login: le pega
    al endpoint de metadata de univtec (headers de dispositivo inventados) y
    arma la URL del transcoded a partir del 'guid' que devuelve."""
    import requests

    ep = _episode_id(page_url_or_id)
    device = str(uuid.uuid4())
    headers = {
        'Referer': 'https://www.lpfplay.com/',
        'Origin': 'https://www.lpfplay.com',
        'platform': 'web',
        'x-tenant-id': 'lpf',
        'x-device-type': 'web',
        'x-device': f'{device};Win32;10_15_7;1.0.0;Google Chrome',
        'x-profile-id': device,
        'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                       '(KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'),
    }
    url = f'https://insight-api-shared.univtec.com/interface/pages/vod/{ep}'
    if verbose:
        print(f'Metadata: {url}')
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()
    items = r.json()
    if not items:
        raise RuntimeError(f'La API no devolvio datos para el episodio {ep}.')
    data = items[0]

    guid = data.get('guid')
    if not guid:
        blob = str(data.get('extraPosters', '')) + str(data.get('poster', ''))
        m = re.search(r'transcoded/([a-f0-9-]{36})/', blob)
        guid = m.group(1) if m else None
    if not guid:
        raise RuntimeError(f'No pude sacar el guid del video. Respuesta: {data}')

    if verbose:
        print(f"Partido : {data.get('title')}")
        print(f"Duracion: {data.get('duration')}")
        print(f"guid    : {guid}")

    return f'https://lpf.vod.immergo.tv/lpf-immergo/transcoded/{guid}/hls/master.m3u8'


def _format_selector(quality):
    h = {'360p': 360, '720p': 720, '1080p': 1080}.get(quality)
    return 'b' if h is None else f'b[height<={h}]/b'


def download_stream(manifest_url, out_path, quality='720p', referer=None,
                    test=False, verbose=True):
    """Descarga el HLS a out_path con yt-dlp (downloader nativo, sin ffmpeg).
    test=True baja solo un fragmento (para validar la cadena)."""
    import yt_dlp

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    tmpl = os.path.splitext(out_path)[0] + '.%(ext)s'
    ydl_opts = {
        'format': _format_selector(quality),
        'outtmpl': tmpl,
        'downloader': 'native',
        'hls_use_mpegts': True,
        'quiet': not verbose,
        'no_warnings': True,
        'noprogress': not verbose,
        'http_headers': {'Referer': referer or 'https://www.lpfplay.com/'},
    }
    if test:
        ydl_opts['test'] = True
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(manifest_url, download=True)
        return ydl.prepare_filename(info)


def download_match_video(page_url, match_key, season=None, quality='720p',
                         referer='https://www.lpfplay.com/', test=False, verbose=True):
    """link del player (o .m3u8 directo) -> resuelve el manifest -> descarga en
    data/temporadas/{season}/partidos/{match_key}/video/."""
    if page_url.endswith('.m3u8') or '.m3u8?' in page_url or '.m3u8#' in page_url:
        manifest = page_url
        if verbose:
            print(f'Usando manifest directo: {manifest[:120]}')
    else:
        manifest = get_master_manifest(page_url, verbose=verbose)

    video_dir = os.path.join(config.season_data_dir(season or config.CURRENT_SEASON),
                             match_key, 'video')
    out_path = os.path.join(video_dir, f'{match_key}.mp4')
    if verbose:
        print(f'Descargando {quality} -> {out_path}')
    final = download_stream(manifest, out_path, quality=quality, referer=referer,
                            test=test, verbose=verbose)
    if verbose:
        size = os.path.getsize(final) / (1024 ** 2) if os.path.exists(final) else 0
        print(f'Video guardado: {final}  ({size:.0f} MB)')
    return final


print('descargador listo: get_master_manifest() / download_match_video()')

descargador listo: get_master_manifest() / download_match_video()


## 1. Resolver la URL del stream

Sin abrir navegador: le pega a la API de metadata y arma el `master.m3u8`.

In [3]:
if MANIFEST_MANUAL.strip():
    manifest = MANIFEST_MANUAL.strip()
    print('Usando MANIFEST_MANUAL:\n', manifest)
else:
    manifest = get_master_manifest(PAGE_URL, verbose=True)

print(f'\n🟢 Stream:\n{manifest}')

Metadata: https://insight-api-shared.univtec.com/interface/pages/vod/6a9dd6070fcb4d947e0c7751
Partido : Fecha 28: Chaco For Ever vs Estudiantes
Duracion: 02:20:15
guid    : 4afccdff-0bae-4ad6-b193-3bec5ef2a390

🟢 Stream:
https://lpf.vod.immergo.tv/lpf-immergo/transcoded/4afccdff-0bae-4ad6-b193-3bec5ef2a390/hls/master.m3u8


## 2. (Opcional) Prueba rápida — 1 fragmento

Validá la cadena bajando un solo fragmento. Si crea un `.mp4` chiquito, todo anda.

In [4]:
prueba = download_match_video(manifest, MATCH_KEY + '_PRUEBA', season=SEASON,
                              quality=QUALITY, test=True, verbose=True)
print('OK' if os.path.exists(prueba) else 'FALLO')
if os.path.exists(prueba):
    os.remove(prueba)
    print('(archivo de prueba eliminado)')

Usando manifest directo: https://lpf.vod.immergo.tv/lpf-immergo/transcoded/4afccdff-0bae-4ad6-b193-3bec5ef2a390/hls/master.m3u8
Descargando 1080p -> c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\data\temporadas\2026\partidos\2026-09-06_vs_estudiantes_caseros_h_PRUEBA\video\2026-09-06_vs_estudiantes_caseros_h_PRUEBA.mp4
[generic] Extracting URL: https://lpf.vod.immergo.tv/lpf-immergo/transcoded/4afccdff-0bae-4ad6-b193-3bec5ef2a390/hls/master.m3u8
[generic] master: Downloading webpage
[generic] master: Downloading m3u8 information
[generic] master: Checking m3u8 live status
[info] master: Downloading 1 format(s): 10800
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1403
[download] Destination: c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\data\temporadas\2026\partidos\2026-09-06_vs_estudiantes_caseros_h_PRUEBA\video\2026-09-06_vs_estudiantes_caseros_h_PRUEBA.mp4
[download] 100% of    1.19MiB in 00:00:00 at 10.25MiB/s                   
Video 

## 3. Descargar el partido completo

Baja el video entero. **Puede tardar** (1080p = varios GB). Al terminar queda
listo para `analisis_video.ipynb`.

In [5]:
video_path = download_match_video(manifest, MATCH_KEY, season=SEASON,
                                  quality=QUALITY, verbose=True)
print(f'\n✅ Listo. Corre analisis_video.ipynb con:')
print(f'   SEASON    = "{SEASON}"')
print(f'   MATCH_KEY = "{MATCH_KEY}"')

Usando manifest directo: https://lpf.vod.immergo.tv/lpf-immergo/transcoded/4afccdff-0bae-4ad6-b193-3bec5ef2a390/hls/master.m3u8
Descargando 1080p -> c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\data\temporadas\2026\partidos\2026-09-06_vs_estudiantes_caseros_h\video\2026-09-06_vs_estudiantes_caseros_h.mp4
[generic] Extracting URL: https://lpf.vod.immergo.tv/lpf-immergo/transcoded/4afccdff-0bae-4ad6-b193-3bec5ef2a390/hls/master.m3u8
[generic] master: Downloading webpage
[generic] master: Downloading m3u8 information
[generic] master: Checking m3u8 live status
[info] master: Downloading 1 format(s): 10800
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1403
[download] Destination: c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\data\temporadas\2026\partidos\2026-09-06_vs_estudiantes_caseros_h\video\2026-09-06_vs_estudiantes_caseros_h.mp4
[download] 100% of    5.70GiB in 00:02:11 at 44.22MiB/s                    
Video guardado: c:\Users\luart\On